In [ ]:
import pandas as pd
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

csv = pd.read_csv('queue.csv')

def _gpus_per_node_from_tres(tres):
    if pd.isna(tres):
        return 0
    s = str(tres)
    # handle ranges like gpu:0-3 -> 4
    m = re.search(r'gpu[:=]\s*(\d+)-(\d+)', s, flags=re.I)
    if m:
        return int(m.group(2)) - int(m.group(1)) + 1
    # handle counts like gres/gpu:4 or gpu:4 or gpu=4
    m = re.search(r'(?:gres/)?gpu[:=]\s*(\d+)', s, flags=re.I)
    if m:
        return int(m.group(1))
    # fallback: any 'gpu' followed by a number
    m = re.search(r'gpu\D*(\d+)', s, flags=re.I)
    if m:
        return int(m.group(1))
    return 0

csv['GPUS'] = csv['TRES_PER_NODE'].apply(_gpus_per_node_from_tres).fillna(0).astype(int) * csv['NODES'].fillna(1).astype(int)

print(csv.head())

In [ ]:
running_jobs = csv[
    (csv['NODELIST'] != '')
    & (csv['STATE'].isin(['COMPLETING', 'RUNNING']))
    & (csv['squeue_id'] == 1)
    & (csv['PARTITION'] == 'boost_usr_prod')
]
print(running_jobs)

In [ ]:
LEONARDO_BOOSTER_SPECS = {
    "num_nodes": 3456,
    "cpus_per_node": 32,
    "gpus_per_node": 4,
    "mem_per_node_gb": 512,
}

SYSTEM_SPECS = LEONARDO_BOOSTER_SPECS

In [ ]:
def expand_nodelist(nodelist):
    """
    Expand Slurm-style nodelist strings.
    Example: 'nid[001-003,007]' -> ['nid001', 'nid002', 'nid003', 'nid007']
    """
    if "[" not in nodelist:
        return [nodelist]

    prefix = nodelist.split("[")[0]
    inside = nodelist.split("[")[1].rstrip("]")

    nodes = []
    for part in inside.split(","):
        if "-" in part:
            start, end = part.split("-")
            width = len(start)
            for i in range(int(start), int(end) + 1):
                nodes.append(f"{prefix}{i:0{width}d}")
        else:
            nodes.append(f"{prefix}{part}")
    return nodes

def all_nodes(prefix="nid", start=0, count=SYSTEM_SPECS["num_nodes"], width=4):
    return [f"{prefix}{i:0{width}d}" for i in range(start, start + count)]

In [ ]:
records = []

for _, row in running_jobs.iterrows():
    nodes = expand_nodelist(row["NODELIST"])
    cpus_per_node = row["CPUS"] / row["NODES"]
    gpus_per_node = row["GPUS"] / row["NODES"]

    for n in nodes:
        records.append(
            {
                "node": n,
                "cpus_used": cpus_per_node,
                "gpus_used": gpus_per_node,
            }
        )

node_df = pd.DataFrame(records)

# -----------------------------
# Aggregate per node
# -----------------------------
node_usage = (
    node_df.groupby("node")
    .agg(
        cpus_used=("cpus_used", "sum"),
        gpus_used=("gpus_used", "sum"),
    )
    .reset_index()
)

# -----------------------------
# Cluster-wide statistics
# -----------------------------
total_cpus = SYSTEM_SPECS["num_nodes"] * SYSTEM_SPECS["cpus_per_node"]
total_gpus = SYSTEM_SPECS["num_nodes"] * SYSTEM_SPECS["gpus_per_node"]

used_cpus = node_usage["cpus_used"].sum()
used_gpus = node_usage["gpus_used"].sum()

stats = {
    "CPU utilization (%)": 100 * used_cpus / total_cpus,
    "GPU utilization (%)": 100 * used_gpus / total_gpus,
    "Active nodes": node_usage["node"].nunique(),
    "Idle nodes": SYSTEM_SPECS["num_nodes"] - node_usage["node"].nunique(),
}

print("=== Cluster statistics ===")
for k, v in stats.items():
    print(f"{k:25s}: {v}")

# -----------------------------
# Visualization
# -----------------------------
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# --- CPU/GPU utilization bars ---
axes[0].bar(
    ["CPUs", "GPUs"],
    [used_cpus / total_cpus, used_gpus / total_gpus],
)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Utilization fraction")
axes[0].set_title("Cluster utilization")

# --- Active vs idle nodes ---
axes[1].bar(
    ["Active", "Idle"],
    [
        stats["Active nodes"],
        stats["Idle nodes"],
    ],
)
axes[1].set_ylabel("Number of nodes")
axes[1].set_title("Node allocation")
axes[1].grid(True, axis='y', alpha=0.5)

# --- Per-node CPU usage distribution ---
axes[2].hist(
    node_usage["cpus_used"],
    bins=np.arange(0, SYSTEM_SPECS["cpus_per_node"] + 1),
    rwidth=0.8,
)
axes[2].set_xlabel("CPUs used per node")
axes[2].set_ylabel("Node count")
axes[2].set_title("CPU usage distribution")
axes[2].grid(True, axis='y', alpha=0.5)

# --- Per-node GPU usage distribution ---
axes[3].hist(
    node_usage["gpus_used"],
    bins=np.arange(0, SYSTEM_SPECS["gpus_per_node"] + 1),
    rwidth=0.8,
)
axes[3].set_xlabel("GPUs used per node")
axes[3].set_ylabel("Node count")
axes[3].set_title("GPU usage distribution")
axes[3].grid(True, axis='y', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
records = []

for _, row in running_jobs.iterrows():
    nodes = expand_nodelist(row["NODELIST"])
    cpus_per_node = row["CPUS"] / row["NODES"]

    for n in nodes:
        records.append(
            {
                "node": n,
                "job_id": row["JOBID"],
                "cpus_used": cpus_per_node,
            }
        )

node_job_df = pd.DataFrame(records)

CPUS_PER_NODE = SYSTEM_SPECS["cpus_per_node"]

node_job_df["cpu_frac"] = (
    node_job_df["cpus_used"] / CPUS_PER_NODE
)

stack_df = (
    node_job_df
    .pivot_table(
        index="node",
        columns="job_id",
        values="cpu_frac",
        aggfunc="sum",
        fill_value=0.0,
    )
    .sort_index()
)

# Reindex to include all nodes - FIX: preserve string index type
all_nodes_list = all_nodes(prefix='lrdn', width=4)[:1000] # FIXME takes time to render 3k nodes
stack_df = stack_df.reindex(all_nodes_list, fill_value=0.0)

if stack_df.shape[1] == 0:
    print("No active jobs to plot!")
else:
    fig, ax = plt.subplots(figsize=(25, 4))

    nodes = stack_df.index
    jobs = stack_df.columns
    data = stack_df.values  # shape: (nodes, jobs)

    x = np.arange(len(nodes))
    width = 0.8
    colors = plt.get_cmap("tab20").colors

    # Cumulative sum along jobs to stack bars
    bottom = np.zeros(len(nodes))
    
    # Create a 2D array where each column is a job's contribution
    job_data = []
    job_labels = []
    
    for i, job_id in enumerate(jobs):
        if (data[:, i] > 0.0).any():
            job_data.append(data[:, i])
            job_labels.append(f"Job {job_id}")
    
    if job_data:
        # Use imshow for much faster rendering with many nodes
        job_array = np.column_stack(job_data)
        
        # Create stacked visualization using imshow
        cumsum_data = np.zeros((len(nodes), len(job_data) + 1))
        for i in range(len(job_data)):
            cumsum_data[:, i+1] = cumsum_data[:, i] + job_data[i]
        
        # Plot each job as a filled area between cumulative sums
        for i in range(len(job_data)):
            ax.fill_between(
                x,
                cumsum_data[:, i],
                cumsum_data[:, i+1],
                color=colors[i % len(colors)],
                label=job_labels[i],
                step='mid',
                linewidth=0
            )

    # Axes
    step = max(1, len(nodes) // 20)
    ax.set_xticks(x[::step])
    ax.set_xticklabels(nodes[::step], rotation=90)
    ax.set_xlim(-0.5, len(nodes) - 0.5)
    ax.set_ylabel("CPU fraction per node")
    ax.set_xlabel("Node")
    ax.set_title("Node allocation by job (CPU fraction)")

    # ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="Jobs")
    plt.tight_layout()
    plt.show()